# ATIVIDADE INTEGRADORA – RELATÓRIO OPERACIONAL DE PRÉ-DECOLAGEM

Este notebook realiza uma análise completa dos dados de telemetria de uma espaçonave, verificando se cada registro presente no CSV atende aos critérios de segurança necessários para a decolagem.

### Integrantes: 
- Allan Victor Santos de Almeida Jesus (RM573218)
- João Pedro de Almeida Freitas (RM572733)
- José Elias Aleixo Lopes (RM568858)
- Sarah Mendes Machado De Oliveira (RM570514)
- Veloso (RM)

## 1. Importar Bibliotecas Necessárias

In [1]:
import pandas as pd
print('importação concluída')

importação concluída


## 2. Carregar Dados de Telemetria da Missão

In [2]:
dados = pd.read_csv('telemetria_missao.csv')

capacidade_total_kwh = 1000  # capacidade total da bateria da nave
consumo_decolagem_kwh = 350  # consumo estimado para a decolagem
perda_percentual = 0.05  # perda percentual de energia durante a decolagem

print('leitura do arquivo concluída')

leitura do arquivo concluída


## 3. Definir Função de Verificação dos Critérios

In [3]:
# Função para verificar os critérios de decolagem
def verificar_registro(row):

    # Nomes dos critérios avaliados para exibição amigavel e alinhada com os dados do registro
    criterios_nomes = [
        'Temp Int (°C)',
        'Temp Ext (°C)',
        'Integridade Estrutural',
        'Energia (%)',
        'Energia para Decolagem (KWh)',
        'Pressão (PSI)',
        'Módulos Críticos'
    ]

    # Avaliação dos critérios de decolagem
    criterios = [
        #exemplo: a temperatura interna deve estar entre 15 e 28 graus Celsius para garantir condições ideais de operação dos sistemas internos da nave
        row['temperatura_interna_celsius'] >= 15 and row['temperatura_interna_celsius'] <= 28,
        row['temperatura_externa_celsius'] >= -50 and row['temperatura_externa_celsius'] <= 60,
        row['integridade_estrutural'] == True,
        row['nivel_energia_percentual'] >= 60 and row['nivel_energia_percentual'] <= 100,
        row['energia_disponivel_kwh'] >= consumo_decolagem_kwh,
        row['pressao_tanques_psi'] >= 80 and row['pressao_tanques_psi'] <= 150,
        row['status_modulos_criticos'] == True
    ]
    
    # Identificação de falhas para exibição
    falhas = [nome for nome, criterio in zip(criterios_nomes, criterios) if not criterio]
    
    # Decisão final baseada nos critérios e respectiva listagem de falhas
    decisao = "PRONTO PARA DECOLAR" if all(criterios) else "DECOLAGEM ABORTADA"
    falhas_str = ', '.join(falhas) if falhas else 'Nenhum'
    
    # Retorna a decisão e as falhas identificadas após a avaliação de todos os critérios de todos os registros
    return {'decisao': decisao, 'falhas': falhas_str}

print('definição da função de verificação concluída')

definição da função de verificação concluída


## 4. Criação de colunas Energia Atual, Energia Disponível e Perda Energética

In [4]:
# Calcular energia antes de verificar cada registro
dados['energia_atual_kwh'] = (dados['nivel_energia_percentual'] / 100) * capacidade_total_kwh
dados['energia_disponivel_kwh'] = dados['energia_atual_kwh'] * (1 - perda_percentual)
dados['perda_energetica_kwh'] = dados['energia_atual_kwh'] * perda_percentual

## 5. Criação de Colunas Decisão e Falhas

As linhas abaixo são responsáveis pela criação das colunas Decisão e Falhas respectivamente.

### Como isso é feito?

A chamada da função verificar_registro é realizada para todas as linhas contidas no DataFrame 'dados' através do atributo '.apply'.

O parâmetro 'axis=1' informado ao final da linha serve para informar que os registros enviados para a função serão passados em linha, e não em coluna.

Os resultados de todos os registros do DataFrame são armazenados em formato de Dictionary na coluna 'resultado' e posteriormente são acessados para a criação das colunas 'decisao' e 'falhas'.

In [5]:
dados['resultado'] = dados.apply(verificar_registro, axis=1)
dados['decisao'] = dados['resultado'].apply(lambda x: x['decisao'])
dados['falhas'] = dados['resultado'].apply(lambda x: x['falhas'])

# Exibição dos resultados apenas para visualização
print(next(iter(dados['resultado'].head())))

{'decisao': 'PRONTO PARA DECOLAR', 'falhas': 'Nenhum'}


## 6. Contagem de Registros Aptos Para Decolagem

In [6]:
# Soma dos registros aprovados e total de registros
aprovados = sum(dados['decisao'] == "PRONTO PARA DECOLAR")
total = len(dados)

print(f"Registros aprovados para decolagem: {aprovados}")
print(f"Total de registros: {total}")

Registros aprovados para decolagem: 15
Total de registros: 30


## 7. Montagem do Cabeçalho e Estatísticas Resumidas

In [7]:
# Montar o Cabeçalho
print("\n" + "="*200)
print(" "*75 + "SISTEMA DE VERIFICAÇÃO DE DECOLAGEM DA MISSÃO")
print("="*200)

# Calcular e Exibir Estatísticas resumidas
print(f"\nResumo:")
print(f"  Total de registros analisados: {total}")
print(f"  ✓ Pronto para decolar: {aprovados} ({100*aprovados/total:.1f}%)")
print(f"  ✗ Decolagem abortada: {total-aprovados} ({100*(total-aprovados)/total:.1f}%)")


                                                                           SISTEMA DE VERIFICAÇÃO DE DECOLAGEM DA MISSÃO

Resumo:
  Total de registros analisados: 30
  ✓ Pronto para decolar: 15 (50.0%)
  ✗ Decolagem abortada: 15 (50.0%)


## 8. Montagem da Tabela Com Detalhes Sobre Cada Registro

In [8]:
# Criar uma tabela formatada para exibição dos detalhes dos registros com os critérios avaliados, decisão e falhas
tabela = dados[[
    'temperatura_interna_celsius', 
    'temperatura_externa_celsius', 
    'integridade_estrutural', 
    'nivel_energia_percentual', 
    'energia_atual_kwh', 
    'energia_disponivel_kwh', 
    'perda_energetica_kwh',
    'pressao_tanques_psi', 
    'status_modulos_criticos', 
    'decisao', 
    'falhas'
    ]].copy()

# Renomear as colunas para exibição mais amigável e alinhada com os critérios avaliados, decisão e falhas identificadas
tabela.columns = [
        'Temp Int (°C)', 
        'Temp Ext (°C)', 
        'Integridade Estrutural', 
        'Energia (%)', 
        'Energia Atual (KWh)', 
        'Energia Disponível (KWh)', 
        'Perda Energética (KWh)', 
        'Pressão (PSI)', 
        'Módulos Críticos', 
        'Decisão', 
        'Falhas'
    ]

## 9. Impressão da Tabela Completa

In [9]:
# Exibir detalhes de todos os registros
print(f"\n{'-'*200}")
print("Detalhamento de todos os registros:")
print(f"{'-'*200}\n")

# Usar pandas display com formatting para exibir a tabela completa de forma legível, garantindo que todas as linhas e colunas sejam mostradas sem truncamento
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)

# Exibir a tabela formatada com os detalhes dos registros, critérios avaliados, decisão e falhas identificadas
print(tabela.to_string())

print(f"\n{'='*200}\n")


--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Detalhamento de todos os registros:
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

    Temp Int (°C)  Temp Ext (°C)  Integridade Estrutural  Energia (%)  Energia Atual (KWh)  Energia Disponível (KWh)  Perda Energética (KWh)  Pressão (PSI)  Módulos Críticos              Decisão                                                       Falhas
0            20.0          -12.8                    True        60.21                602.1                   571.995                  30.105         120.00              True  PRONTO PARA DECOLAR                                                       Nenhum
1            25.0            0.2                